# 7 · Day 4 – Monitoring refresh
**Job task `dashboard_refresh`** – last task in the job. Rebuilds the monitoring tables that the dashboard reads.

| Table (schema `ops`) | Mode | What it shows |
|---|---|---|
| `monitor_table_freshness` | overwrite | every pipeline table: rows, last write time, hours since last write |
| `monitor_rejected_records` | overwrite | rejected records by source table and DQ rule, with reject % |
| `monitor_pipeline_runs` | append | one row per job run: checks passed / warned / failed, overall status |

## 1. Setup

This cell gets the notebook ready:
* Reads the `catalog` and `run_id` values from the job (or uses the defaults when you run it by hand).
* Sets the schema names for bronze, silver, gold and ops, and creates the `ops` schema if it does not exist.
* Defines two small helpers: `count()` returns how many rows a table has, and `hours_since_last_write()` returns how many hours have passed since the table was last written.

In [0]:
from pyspark.sql import functions as F

# Job parameters (the job fills these in; defaults are used when you run the notebook by hand)
dbutils.widgets.text("catalog", "capstone_5iot")
dbutils.widgets.text("run_id", "manual")
CATALOG = dbutils.widgets.get("catalog")
RUN_ID  = dbutils.widgets.get("run_id")

BRONZE = f"{CATALOG}.iot_schema"
SILVER = f"{CATALOG}.silver"
GOLD   = f"{CATALOG}.gold"
OPS    = f"{CATALOG}.ops"            # pipeline monitoring tables live here
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {OPS}")

def count(table):
    return spark.table(table).count()

def hours_since_last_write(table):
    ts  = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["timestamp"]
    now = spark.sql("SELECT current_timestamp()").first()[0]   # Spark clock, same time zone as ts
    return round((now - ts).total_seconds() / 3600, 2)

print(f"catalog={CATALOG}  run_id={RUN_ID}")

## 2. Table freshness

This cell checks how fresh each pipeline table is:
* Lists all bronze, silver and gold tables.
* For each table, gets the row count, the last write time, the last operation, and the hours since the last write.
* Saves the result to `ops.monitor_table_freshness` (replacing the old data) and shows it.

In [0]:
TABLES = {
    "bronze": [f"{BRONZE}.{t}" for t in ["bronze_machine_master", "bronze_plant_master", "bronze_failure_code_lookup",
                                         "bronze_sensor_readings", "bronze_failure_events", "bronze_work_orders"]],
    "silver": [f"{SILVER}.{t}" for t in ["silver_sensor_readings", "silver_failure_events", "silver_work_orders",
                                         "silver_rejected_records", "silver_maintenance_episodes"]],
    "gold":   [f"{GOLD}.{t}"   for t in ["dim_machine", "dim_plant", "fact_sensor_activity",
                                         "fact_maintenance_episodes", "mart_sensor_daily"]],
}

rows = []
for layer, tables in TABLES.items():
    for t in tables:
        last = spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").first()
        rows.append((layer, t, count(t), last["timestamp"], last["operation"], hours_since_last_write(t)))

freshness = (spark.createDataFrame(rows, "layer string, table_name string, row_count long, last_write_ts timestamp, "
                                         "last_operation string, hours_since_write double")
                  .withColumn("refreshed_ts", F.current_timestamp()))
freshness.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OPS}.monitor_table_freshness")
display(freshness)

## 3. Rejected records by rule

This cell summarises the records that failed data quality checks:
* Counts the rows in each bronze source table.
* Counts rejected rows in `silver_rejected_records` by source table and rule.
* Works out the reject % (rejected rows divided by bronze rows).
* Saves the result to `ops.monitor_rejected_records` (replacing the old data) and shows it.

In [0]:
bronze_rows = spark.createDataFrame(
    [(t, count(f"{BRONZE}.{t}")) for t in ["bronze_sensor_readings", "bronze_failure_events", "bronze_work_orders"]],
    "source_table string, bronze_rows long")

rejected = (spark.table(f"{SILVER}.silver_rejected_records")
    .groupBy("source_table", "rule_id", "reason").agg(F.count("*").alias("rejected_rows"))
    .join(bronze_rows, "source_table", "left")
    .withColumn("reject_pct", F.round(100 * F.col("rejected_rows") / F.col("bronze_rows"), 2))
    .withColumn("refreshed_ts", F.current_timestamp()))
rejected.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{OPS}.monitor_rejected_records")
display(rejected.orderBy("source_table", "rule_id"))

## 4. Run history (one row per job run)

This cell records one summary row for the current job run:
* Reads this run's validation results and counts how many checks passed, warned and failed.
* Sets the overall status: FAILED if any check failed, WARNING if any check warned, otherwise SUCCESS.
* Adds the row counts of the rejected records, fact and mart tables.
* Appends the row to `ops.monitor_pipeline_runs` (old runs are kept) and shows the full history, newest first.

In [0]:
checks = spark.table(f"{OPS}.pipeline_validation_results").filter(F.col("run_id") == RUN_ID)
c = {r["status"]: r["n"] for r in checks.groupBy("status").agg(F.count("*").alias("n")).collect()}

run_row = spark.createDataFrame([(
        RUN_ID, spark.sql("SELECT current_timestamp()").first()[0],
        c.get("PASS", 0), c.get("WARN", 0), c.get("FAIL", 0),
        "FAILED" if c.get("FAIL", 0) else ("WARNING" if c.get("WARN", 0) else "SUCCESS"),
        count(f"{SILVER}.silver_rejected_records"),
        count(f"{GOLD}.fact_sensor_activity"),
        count(f"{GOLD}.mart_sensor_daily"))],
    "run_id string, run_ts timestamp, checks_passed long, checks_warned long, checks_failed long, "
    "overall_status string, rejected_records long, fact_sensor_rows long, mart_rows long")

run_row.write.mode("append").saveAsTable(f"{OPS}.monitor_pipeline_runs")
display(spark.table(f"{OPS}.monitor_pipeline_runs").orderBy(F.desc("run_ts")))